KENNICHE Redouane


# Deep Learning pour la Prédiction de Types de Cancer

L'objectif de ce projet est d'utiliser les réseaux de neurones artificiels pour prédire le type de cancer d'un patient.

**Le défi clinique et technique :** Notre jeu de données est de petite taille (environ 600 patients) et très déséquilibré. Les réseaux de neurones étant extrêmement puissants, ils ont tendance à "apprendre par cœur" les petits jeux de données (surapprentissage) au lieu de généraliser. Notre démarche se concentrera donc massivement sur la **régularisation** (Dropout, Early Stopping) et la **gestion du déséquilibre** (Poids dans la fonction de perte) pour garantir un modèle cliniquement fiable.

In [1]:
# ==============================================================================
# 1. IMPORTS ET PRÉPARATION DES DONNÉES
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

#On fixe d'abord l'aléatoire pour pouvoir reproduire les résultats
import random
seed = 50

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Configuration de l'appareil (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Chargement
train_df = pd.read_excel('donnee_etu.xlsx', sheet_name='train')
validation_df = pd.read_excel('donnee_etu.xlsx', sheet_name='Validation')

X_train_raw = train_df.drop(columns=['Type'])
# En PyTorch : Les classes doivent commencer à 0 (donc Type 1 devient 0, Type 6 devient 5)
y_train_raw = train_df['Type'].values - 1
X_val_raw = validation_df.drop(columns=['Type']).copy()

# Preprocessing (pour éviter le data leakage)
mapping = {'POS': 1, 'NEG': 0}
categorical_cols = ['marqueur 2', 'marqueur 3', 'marqueur 4', 'marqueur 5']
for col in categorical_cols:
    X_train_raw[col] = X_train_raw[col].map(mapping)
    X_val_raw[col] = X_val_raw[col].map(mapping)

numeric_features = ['age', 'marqueur 6', 'marqueur 7', 'marqueur 8', 'marqueur 9']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Application du preprocessing
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_val_processed = preprocessor.transform(X_val_raw)

# Calcul des poids des classes pour gérer le déséquilibre
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_raw), y=y_train_raw)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Création du Dataset PyTorch
class CancerDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

## 1. Définition des Architectures et Stratégie

Pour répondre à la consigne, nous allons comparer 3 modèles de réseaux de neurones (Perceptrons Multicouches - MLP) de complexité croissante :
1. **Modèle "Shallow" (Peu profond) :** 1 seule couche cachée. Souvent le plus robuste sur les petits jeux de données tabulaires.
2. **Modèle "Deep" (Profond) :** 3 couches cachées. Il a une plus grande capacité d'abstraction mais risque de surapprendre.
3. **Modèle "Regularized" (Profond + Dropout) :** 3 couches cachées avec des couches de `Dropout`. Le Dropout "désactive" aléatoirement des neurones pendant l'entraînement, forçant le réseau à ne pas se reposer sur un seul marqueur biologique. C'est notre bouclier anti-surapprentissage.

**Argumentaire clinique :** Nous utiliserons la fonction de perte `CrossEntropyLoss` en y injectant les poids calculés des classes (`class_weights`). Cela force le réseau à pénaliser beaucoup plus lourdement une erreur faite sur un cancer rare (Types 2 et 3) que sur un cancer fréquent.

In [2]:
# ==============================================================================
# 2. DÉFINITION DES ARCHITECTURES DU RÉSEAU DE NEURONES
# ==============================================================================
class CancerNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=1, dropout_rate=0.0):
        super(CancerNN, self).__init__()
        layers = []

        # Couche d'entrée
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))

        # Couches cachées
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))

        # Couche de sortie
        layers.append(nn.Linear(hidden_dim, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Les 3 architectures à tester
configs = {
    'Shallow (1 couche)': {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.0},
    'Deep (3 couches)': {'hidden_dim': 32, 'num_layers': 3, 'dropout': 0.0},
    'Deep + Dropout (Régularisé)': {'hidden_dim': 32, 'num_layers': 3, 'dropout': 0.3}
}
learning_rates = [0.01, 0.001]

## 2. Protocole de Fine-Tuning et Hyperparamètres

La rigueur de l'évaluation est assurée par le protocole suivant :
* **Validation Croisée Stratifiée (5-Fold) :** Pour maintenir la représentativité des cancers rares dans chaque sous-échantillon d'entraînement et de validation.
* **Early Stopping (Critère d'arrêt) :** Plutôt que d'entraîner le modèle sur un nombre fixe d'époques, nous surveillons le F1-Score sur le pli de validation à chaque itération. Si la performance stagne ou chute pendant 15 époques consécutives (signe que le modèle commence à apprendre par cœur), l'entraînement s'arrête prématurément.
* **Espace des hyperparamètres :** Pour chaque architecture, nous testerons différents taux d'apprentissage (*Learning Rate*). La métrique d'optimisation reste le **F1-Score Macro**, indispensable en santé pour les données déséquilibrées.

In [3]:
# ==============================================================================
# 3. BOUCLE D'ENTRAÎNEMENT, EARLY STOPPING ET VALIDATION CROISÉE
# ==============================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
num_epochs = 150
patience = 15 # Nombre d'époques sans amélioration avant l'Early Stopping
input_dim = X_train_processed.shape[1]
num_classes = 6

best_global_score = 0
best_model_name = ""
best_lr = 0

print("Démarrage de la recherche d'hyperparamètres et des architectures...\n")

for name, params in configs.items():
    for lr in learning_rates:
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_processed, y_train_raw)):
            # Création des DataLoaders pour ce Pli (Fold)
            train_sub_ds = CancerDataset(X_train_processed[train_idx], y_train_raw[train_idx])
            val_sub_ds = CancerDataset(X_train_processed[val_idx], y_train_raw[val_idx])

            train_loader = DataLoader(train_sub_ds, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_sub_ds, batch_size=32, shuffle=False)

            # Initialisation du modèle
            model = CancerNN(input_dim, params['hidden_dim'], num_classes, params['num_layers'], params['dropout']).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) # Gestion du déséquilibre !
            optimizer = optim.Adam(model.parameters(), lr=lr)

            best_val_f1 = 0
            epochs_no_improve = 0

            # Entraînement avec Early Stopping
            for epoch in range(num_epochs):
                model.train()
                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    loss.backward()
                    optimizer.step()

                # Évaluation sur le pli de validation
                model.eval()
                all_preds = []
                all_targets = []
                with torch.no_grad():
                    for X_val_batch, y_val_batch in val_loader:
                        X_val_batch = X_val_batch.to(device)
                        outputs = model(X_val_batch)
                        _, preds = torch.max(outputs, 1)
                        all_preds.extend(preds.cpu().numpy())
                        all_targets.extend(y_val_batch.numpy())

                val_f1 = f1_score(all_targets, all_preds, average='macro')

                # Mécanique de l'Early Stopping
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

                if epochs_no_improve >= patience:
                    break # On arrête l'entraînement pour éviter le surapprentissage

            fold_scores.append(best_val_f1)

        mean_f1 = np.mean(fold_scores)
        print(f"Modèle: {name} | LR: {lr} | F1-Macro CV: {mean_f1:.4f}")

        if mean_f1 > best_global_score:
            best_global_score = mean_f1
            best_model_name = name
            best_lr = lr
            best_params = params

print(f"\n🏆 Meilleur Modèle : {best_model_name} avec un LR de {best_lr} (F1-Macro = {best_global_score:.4f})")

Démarrage de la recherche d'hyperparamètres et des architectures...

Modèle: Shallow (1 couche) | LR: 0.01 | F1-Macro CV: 0.8316
Modèle: Shallow (1 couche) | LR: 0.001 | F1-Macro CV: 0.8025
Modèle: Deep (3 couches) | LR: 0.01 | F1-Macro CV: 0.8157
Modèle: Deep (3 couches) | LR: 0.001 | F1-Macro CV: 0.8102
Modèle: Deep + Dropout (Régularisé) | LR: 0.01 | F1-Macro CV: 0.8125
Modèle: Deep + Dropout (Régularisé) | LR: 0.001 | F1-Macro CV: 0.7893

🏆 Meilleur Modèle : Shallow (1 couche) avec un LR de 0.01 (F1-Macro = 0.8316)


## 3. Entraînement Final et Prédictions

Maintenant que notre protocole de validation croisée nous a permis d'identifier l'architecture la plus robuste (le modèle "Shallow") et les meilleurs hyperparamètres (Learning Rate = 0.01), nous passons à la phase de prédiction :

1. **Entraînement final :** Nous ré-instancions ce modèle optimal et l'entraînons cette fois sur **l'intégralité** de notre jeu de données d'entraînement. Puisque nous utilisons 100% des données, nous ne faisons plus d'Early Stopping mais fixons un nombre d'époques raisonnable (50) observé lors de nos expérimentations précédentes.
2. **Prédiction aveugle :** Le modèle final génère les prédictions sur le jeu de validation. Les classes PyTorch (0-5) sont ensuite re-converties au format clinique (1-6) pour l'export.

In [4]:
# ==============================================================================
# 4. ENTRAÎNEMENT FINAL ET PRÉDICTIONS
# ==============================================================================
print("Entraînement du modèle final sur 100% des données...")

# On ré-entraîne la meilleure architecture sur tout le dataset d'entraînement
final_model = CancerNN(input_dim, best_params['hidden_dim'], num_classes, best_params['num_layers'], best_params['dropout']).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(final_model.parameters(), lr=best_lr)

final_train_ds = CancerDataset(X_train_processed, y_train_raw)
final_train_loader = DataLoader(final_train_ds, batch_size=32, shuffle=True)

# Entraînement final (sans early stopping ici puisqu'on utilise toutes les données, on fixe une limite raisonnable)
final_model.train()
for epoch in range(50):
    for X_batch, y_batch in final_train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = final_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

# Prédictions sur le jeu de validation
val_ds = CancerDataset(X_val_processed)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

final_model.eval()
final_predictions = []
with torch.no_grad():
    for X_batch in val_loader:
        X_batch = X_batch.to(device)
        outputs = final_model(X_batch)
        _, preds = torch.max(outputs, 1)
        final_predictions.extend(preds.cpu().numpy())

# ATTENTION : On rajoute +1 pour repasser des classes PyTorch (0 à 5) aux classes cliniques (1 à 6)
final_predictions = np.array(final_predictions) + 1

# Sauvegarde
validation_df['Type'] = final_predictions
validation_df.to_csv('Validation_DL_KENNICHE_Redouane.csv', index=False)
print("Prédictions terminées. Fichier 'Validation_DL_rempli.csv' généré avec succès.")

Entraînement du modèle final sur 100% des données...
Prédictions terminées. Fichier 'Validation_DL_rempli.csv' généré avec succès.
